# Oracle 26ai Banking Nudges — Self-Contained Training Notebook

This notebook is a single, runnable training artifact for the proactive banking nudges demo on Oracle Database 26ai. It replaces the need to open multiple README files, SQL scripts, and code folders, while still pointing to external assets only where they truly belong outside a notebook (raw dataset downloads, APEX export, and Spring/Java examples).
# What You Will Learn

By the end of this notebook you will understand how to:

- Use Oracle 26ai as one surface for relational, vector, graph, and AI workloads.
- Load and call an ONNX embedding model inside the database.
- Build a property graph overlay on existing relational tables with SQL/PGQ.
- Configure Select AI so natural language can generate SQL against your schema.
- Wire the database to an MCP-compatible agent and to an APEX chat UI.
- Implement three real-time nudge use cases.
- Operate the result with latency, capacity, security, and graceful-degradation guardrails.
# The Business Problem and Target Moments

The demo targets three high-intent banking moments:

1. **Credit card product page view** — assist while the customer's intent is fresh.
2. **Application abandonment** — recover the customer before intent decays.
3. **Declined transaction** — resolve the issue quickly with a concrete next action.

Success condition: decision latency must be low enough to act before the session or context is lost.
# Core Design Principle

**One operational surface.** Keep the source of truth where it already lives. Extend existing tables with vector columns and graph overlays rather than introducing separate vector and graph stores. Execute mixed retrieval in one query path with ACID guarantees. Only add external systems when scale or autonomy requirements are proven.
# The "AI" Is Just New Datatypes and Operators

From a DBA perspective, the AI pieces are concrete database objects:

- `VECTOR` column = fixed-length `FLOAT32` array.
- ONNX model = stored function loaded via `DBMS_VECTOR.LOAD_ONNX_MODEL`.
- Vector index = a new index type for approximate nearest-neighbor search.
- `VECTOR_EMBEDDING(... USING ... AS DATA)` and `VECTOR_DISTANCE(..., ..., COSINE)` = new SQL operators.
- Property Graph = a view-like overlay on relational tables.
- Select AI profile = a `DBMS_CLOUD_AI` package configuration.
- MCP server = a listener that exposes database tools to an LLM.

# Architecture

```mermaid
flowchart TD
    Datasets[Public Datasets: PaySim, LendingClub, Banking77, UCI] --> Scripts[scripts/01..04]
    Scripts --> Obj[OCI Object Storage]
    Obj --> Load[DBMS_CLOUD.COPY_DATA]
    Load --> Staging[STG_* Tables]
    Staging --> Transform[sql/05_transform.sql]
    Transform --> Core[(CUSTOMER ACCOUNT TXN APPLICATION PRODUCT OFFER PAGE_EVENT CONVERSATION)]
    Core --> Vector[CONVERSATION_CHUNK + VECTOR INDEX]
    Core --> Graph[banking_graph via SQL/PGQ]
    Core --> SelectAI[DBMS_CLOUD_AI profile NUDGE_BOT]
    APEX[APEX chat page] --> Core
    MCP[SQLcl MCP server] --> Core
```

# Environment Prerequisites

Before running cells you need:

- Python 3.10+ with `oracledb`, `pandas`, `numpy`, `python-dotenv`, and optionally `langchain-oracledb`.
- Oracle Database 26ai: local Docker/Podman container, Autonomous Database Free Tier, or ADB-S/dedicated.
- For ADB: either a wallet directory with `TNS_ADMIN` pointing to it, or a TLS-only `tnsnames.ora` directory if your ADB supports one-way TLS.
- For model download: `docker`/`podman` for local Oracle, or `DBMS_CLOUD.GET_OBJECT` for ADB.
- For Select AI: an OCI GenAI credential (`OCI_GENAI_CRED`) created in the database.
- For MCP: SQLcl 24+ installed locally.
- For dataset ingestion: Kaggle API key and optionally OCI CLI for object-storage uploads.

Create a `.env` file in the notebook folder if you are not using Codespaces secrets:

```text
ORACLE_USER=testuser
ORACLE_PASSWORD=TestPass123
ORACLE_DSN=localhost:1521/FREEPDB1
ORACLE_MODEL_NAME=MINILM_EMB
ORACLE_ONNX_FILE=all_MiniLM_L6_v2.onnx
ORACLE_DIRECTORY_NAME=ONNX_DIR
BANKING_DEMO_ROOT=/workspaces/oracle-26ai-learning/26ai-banking-demo
TNS_ADMIN=/path/to/wallet
```

# Configure GitHub Codespaces Secrets

When running this notebook in a GitHub Codespace, store credentials as Codespace secrets instead of committing them:

1. Go to your personal **GitHub settings → Codespaces → Secrets**.
2. Add a secret for each value below. They become environment variables when the Codespace starts:

| Secret | Environment variable | Example value |
|---|---|---|
| `ORACLE_USER` | `ORACLE_USER` | `ADMIN` or `TESTUSER` |
| `ORACLE_PASSWORD` | `ORACLE_PASSWORD` | your database password |
| `ORACLE_DSN` | `ORACLE_DSN` | `localhost:1521/FREEPDB1` or `nudgedb_high` |
| `TNS_ADMIN` | `TNS_ADMIN` | `/home/codespace/wallet` |

3. Rebuild or restart the Codespace so the secrets are exported as environment variables.
4. Do not place wallet ZIP files, TLS config directories (for example `wallet_tls/`), `.sso`, `.pem`, `.p12`, `.ora`, or `.env` files inside the repo. They will be blocked by `.gitignore`.

The next cell verifies the secrets are present and masks the password before falling back to `.env`.

# Alternative: Connect with TLS Instead of a Wallet

Oracle Autonomous Database can accept TLS (one-way TLS) connections without requiring a downloaded wallet, depending on your ADB network and security configuration. If TLS is enabled, you can connect using a TLS connection string and skip the `Wallet_*.zip` entirely.

## How to get the TLS connection string

1. In the OCI Console, open your Autonomous Database details page.
2. Click **Database connection**.
3. Under **Connection strings**, switch the **TLS authentication** option to **TLS** instead of **mTLS** if your tenancy/ADB supports it.
4. Copy the **TLS connection string**. It looks similar to:

```text
adbname_high =
  (description=
    (retry_count=20)(retry_delay=3)
    (address=(protocol=tcps)(port=1522)(host=adb.<region>.oraclecloud.com))
    (connect_data=(service_name=<service_name>))
    (security=(ssl_server_cert_dn="CN=adb.<region>.oraclecloud.com")
              (ssl_server_auto_dn_match=yes))
  )
```

## How to use it in this notebook

Create a directory (for example `/home/codespace/wallet_tls`) and place only a `tnsnames.ora` file inside it with the TLS descriptor above. Then set:

```text
TNS_ADMIN=/home/codespace/wallet_tls
ORACLE_DSN=adbname_high
```

`python-oracledb` thin mode uses the operating-system certificate store to validate the Oracle server certificate, so no `cwallet.sso`, `ewallet.p12`, or `sqlnet.ora` is required.

## Caveats

- TLS without a wallet is not available for every Autonomous Database deployment. If the console does not show a TLS option, continue with the wallet.
- Some organizations require mTLS with a wallet for compliance even when TLS is technically available.
- If you see `ORA-28759: failure to open file` or certificate-validation errors, the OS trust store may be missing the required CA. In that case either install the CA into the system store or fall back to the wallet.
- The `TNS_ADMIN` secret is still useful because it points to whichever directory holds your `tnsnames.ora` (wallet or TLS-only).

In [1]:
import os
from pathlib import Path

# Secrets priority: environment variables (GitHub Codespaces) -> .env file -> defaults.
required = ["ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN", "TNS_ADMIN"]
env_dotenv = Path(".env")
if env_dotenv.exists():
    for line in env_dotenv.read_text().splitlines():
        if line.strip() and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

missing = [s for s in required if not os.environ.get(s)]
if missing:
    print("WARNING: missing secrets/env vars:", missing)
    print("Set them via GitHub Codespaces Secrets or add a .env file.")
else:
    print("All required secrets are present.")
    print(f"ORACLE_USER  = {os.environ.get('ORACLE_USER')}")
    print(f"ORACLE_DSN   = {os.environ.get('ORACLE_DSN')}")
    print(f"TNS_ADMIN    = {os.environ.get('TNS_ADMIN')}")
    print(f"ORACLE_PASSWORD = {'*' * len(os.environ.get('ORACLE_PASSWORD', ''))}")

Set them via GitHub Codespaces Secrets or add a .env file.


In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'oracledb', 'pandas', 'numpy', 'python-dotenv',
                'langchain', 'langchain-core', 'langchain-oracledb'],
               check=False)
print("Dependencies installed.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Dependencies installed.


In [3]:
import os
import oracledb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

user = os.environ.get("ORACLE_USER", "testuser")
password = os.environ.get("ORACLE_PASSWORD", "TestPass123")
dsn = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")
model_name = os.environ.get("ORACLE_MODEL_NAME", "MINILM_EMB")
onnx_file = os.environ.get("ORACLE_ONNX_FILE", "all_MiniLM_L6_v2.onnx")
directory_name = os.environ.get("ORACLE_DIRECTORY_NAME", "ONNX_DIR")
demo_root = os.environ.get("BANKING_DEMO_ROOT", "/workspaces/oracle-26ai-learning/26ai-banking-demo")
tns_admin = os.environ.get("TNS_ADMIN")

if tns_admin:
    os.environ["TNS_ADMIN"] = tns_admin

if user in ("testuser", "ADMIN") and password in ("TestPass123", "Welcome12345#"):
    print("WARNING: using default credentials. Set ORACLE_USER/ORACLE_PASSWORD via Codespaces secrets or .env.")

conn = oracledb.connect(user=user, password=password, dsn=dsn)

def run_sql(sql, params=None, fetch=True):
    with conn.cursor() as cur:
        cur.execute(sql, params or {})
        if fetch and cur.description:
            columns = [col[0] for col in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=columns)
        return None

version = run_sql("SELECT * FROM v$version WHERE banner LIKE 'Oracle%'").iloc[0, 0]
print(f"Connected to: {dsn}")
print(f"Oracle version: {version}")

OperationalError: DPY-6005: cannot connect to database (CONNECTION_ID=I++Gd3+VgXjXK+8yrvl08Q==).
[Errno 111] Connection refused

# Download and Stage Public Datasets

Kaggle credentials and multi-gigabyte downloads belong outside the notebook. Run the repo scripts from the terminal:

```bash
./26ai-banking-demo/scripts/00_setup_kaggle.sh
./26ai-banking-demo/scripts/01_download_all.sh
python3 26ai-banking-demo/scripts/02_trim_lending.py \
  --input 26ai-banking-demo/data/raw/lendingclub/accepted_2007_to_2018Q4.csv \
  --output 26ai-banking-demo/data/processed/lendingclub_5k.csv
python3 26ai-banking-demo/scripts/03_gen_conversations.py \
  --input 26ai-banking-demo/data/raw/banking77/banking77.csv \
  --output 26ai-banking-demo/data/processed/banking77_conversations.csv
```

The next cell verifies the expected files exist.

In [ ]:
from pathlib import Path
import pandas as pd

demo_root = Path(os.environ.get("BANKING_DEMO_ROOT", "/workspaces/oracle-26ai-learning/26ai-banking-demo"))
expected = {
    "paysim": demo_root / "data/raw/paysim/PS_20174392719_1491204439457_log.csv",
    "lendingclub": demo_root / "data/processed/lendingclub_5k.csv",
    "banking77": demo_root / "data/processed/banking77_conversations.csv",
    "marketing": demo_root / "data/raw/marketing/bank-additional-full.csv",
}

for name, path in expected.items():
    if path.exists():
        rows = sum(1 for _ in open(path)) - 1
        print(f"{name}: OK ({rows:,} rows) -> {path}")
    else:
        print(f"{name}: MISSING -> {path}")

# Upload to OCI Object Storage (Optional)

If you are using Autonomous Database, upload the processed CSVs to an OCI bucket first:

```bash
OCI_NAMESPACE=<namespace> OCI_BUCKET_NAME=<bucket> ./26ai-banking-demo/scripts/04_upload_to_oci.sh
```

If you are running the local Docker Oracle image, you can skip OCI and load data via external tables or direct `pandas` inserts shown in the fallback path below.

In [ ]:
schema_sql = """
CREATE TABLE customer (
  customer_id    NUMBER PRIMARY KEY,
  full_name      VARCHAR2(120),
  segment        VARCHAR2(40),
  signup_date    DATE
);

CREATE TABLE product (
  product_id     NUMBER PRIMARY KEY,
  name           VARCHAR2(120),
  family         VARCHAR2(40),
  details_blob   BLOB,
  details_text   CLOB
);

CREATE TABLE offer (
  offer_id          NUMBER PRIMARY KEY,
  product_id        NUMBER REFERENCES product(product_id),
  offer_name        VARCHAR2(120),
  eligibility_rule  VARCHAR2(400),
  outcome_label     VARCHAR2(40)
);

CREATE TABLE account (
  account_id     NUMBER PRIMARY KEY,
  customer_id    NUMBER REFERENCES customer(customer_id),
  product_id     NUMBER REFERENCES product(product_id),
  daily_limit    NUMBER,
  opened_at      DATE
);

CREATE TABLE txn (
  txn_id          NUMBER PRIMARY KEY,
  account_id      NUMBER REFERENCES account(account_id),
  amount          NUMBER,
  status          VARCHAR2(20),
  decline_reason  VARCHAR2(80),
  txn_ts          TIMESTAMP
);

CREATE TABLE application (
  app_id         NUMBER PRIMARY KEY,
  customer_id    NUMBER REFERENCES customer(customer_id),
  product_id     NUMBER REFERENCES product(product_id),
  status         VARCHAR2(20),
  fields_json    JSON,
  updated_at     TIMESTAMP
);

CREATE TABLE page_event (
  event_id       NUMBER PRIMARY KEY,
  customer_id    NUMBER REFERENCES customer(customer_id),
  product_id     NUMBER REFERENCES product(product_id),
  page_url       VARCHAR2(400),
  event_ts       TIMESTAMP
);

CREATE TABLE conversation (
  conv_id        NUMBER PRIMARY KEY,
  customer_id    NUMBER REFERENCES customer(customer_id),
  channel        VARCHAR2(20),
  transcript     CLOB,
  conv_ts        TIMESTAMP
);

CREATE TABLE conversation_chunk (
  chunk_id       NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  conv_id        NUMBER REFERENCES conversation(conv_id),
  chunk_text     VARCHAR2(4000),
  embedding      VECTOR(384, FLOAT32)
);
"""

run_sql(schema_sql, fetch=False)
print("Core schema created.")
print("Tables: customer, product, offer, account, txn, application, page_event, conversation, conversation_chunk")

In [ ]:
staging_sql = """
CREATE TABLE stg_paysim (
  step               NUMBER,
  type               VARCHAR2(20),
  amount             NUMBER,
  name_orig          VARCHAR2(40),
  oldbalance_org     NUMBER,
  newbalance_orig    NUMBER,
  name_dest          VARCHAR2(40),
  oldbalance_dest    NUMBER,
  newbalance_dest    NUMBER,
  is_fraud           NUMBER,
  is_flagged_fraud   NUMBER
);

CREATE TABLE stg_lending (
  id               NUMBER,
  member_id        NUMBER,
  loan_amnt        NUMBER,
  term             VARCHAR2(30),
  int_rate         VARCHAR2(20),
  grade            VARCHAR2(5),
  sub_grade        VARCHAR2(5),
  emp_length       VARCHAR2(30),
  home_ownership   VARCHAR2(30),
  annual_inc       NUMBER,
  purpose          VARCHAR2(100),
  loan_status      VARCHAR2(80),
  issue_d          VARCHAR2(20)
);

CREATE TABLE stg_banking77 (
  text             VARCHAR2(500),
  category         VARCHAR2(60)
);

CREATE TABLE stg_marketing (
  age               NUMBER,
  job               VARCHAR2(40),
  marital           VARCHAR2(20),
  education         VARCHAR2(40),
  "default"         VARCHAR2(10),
  housing           VARCHAR2(10),
  loan              VARCHAR2(10),
  contact           VARCHAR2(20),
  month             VARCHAR2(10),
  day_of_week       VARCHAR2(10),
  duration          NUMBER,
  campaign          NUMBER,
  pdays             NUMBER,
  previous          NUMBER,
  poutcome          VARCHAR2(20),
  emp_var_rate      NUMBER,
  cons_price_idx    NUMBER,
  cons_conf_idx     NUMBER,
  euribor3m         NUMBER,
  nr_employed       NUMBER,
  y                 VARCHAR2(10)
);
"""

run_sql(staging_sql, fetch=False)
print("Staging tables created: stg_paysim, stg_lending, stg_banking77, stg_marketing")

In [ ]:
try:
    # Path B: ADB / cloud — pull the model from Oracle's public bucket and load it.
    load_cloud_sql = """
    BEGIN
      DBMS_CLOUD.GET_OBJECT(
        credential_name => NULL,
        object_uri => 'https://objectstorage.us-phoenix-1.oraclecloud.com/n/adwc4pm/b/OML-Resources/o/all_MiniLM_L6_v2.onnx',
        directory_name => 'DATA_PUMP_DIR'
      );
    END;
    """
    run_sql(load_cloud_sql, fetch=False)

    run_sql("""
    BEGIN
      DBMS_VECTOR.LOAD_ONNX_MODEL(
        'DATA_PUMP_DIR',
        'all_MiniLM_L6_v2.onnx',
        'MINILM_EMB',
        JSON('{"function":"embedding","embeddingOutput":"embedding","input":{"input":["DATA"]}}')
      );
    END;
    """, fetch=False)
    print("ONNX model loaded via DBMS_CLOUD.GET_OBJECT.")
except Exception as e:
    print(f"Cloud model load failed (expected if using local Docker or missing privileges): {e}")
    print("For local Docker, download all_MiniLM_L6_v2.onnx, copy it into the container, and load from a custom DIRECTORY.")

models = run_sql("SELECT model_name, mining_function FROM user_mining_models WHERE model_name = 'MINILM_EMB'")
print(models)

# Ingest Staging Data

For ADB, fill in the placeholders in `04_copy_data.sql` and run the cloud copy. The notebook defaults to a local pandas fallback that inserts into staging tables directly, which works for local Docker Oracle without object storage.

In [ ]:
import pandas as pd

def load_csv_to_oracle(path, table, columns=None, chunksize=1000, sep=","):
    if not path.exists():
        print(f"SKIP: {path} not found")
        return 0
    df = pd.read_csv(path, sep=sep, low_memory=False)
    if columns:
        df = df[columns]
    df = df.where(pd.notnull(df), None)
    rows = []
    with conn.cursor() as cur:
        for r in df.itertuples(index=False, name=None):
            rows.append(r)
            if len(rows) >= chunksize:
                cur.executemany(f"INSERT INTO {table} VALUES ({','.join([':' + str(i+1) for i in range(len(rows[0]))])})", rows)
                rows = []
        if rows:
            cur.executemany(f"INSERT INTO {table} VALUES ({','.join([':' + str(i+1) for i in range(len(rows[0]))])})", rows)
    conn.commit()
    print(f"Loaded {len(df)} rows into {table}")
    return len(df)

demo_root = Path(os.environ.get("BANKING_DEMO_ROOT", "/workspaces/oracle-26ai-learning/26ai-banking-demo"))

load_csv_to_oracle(demo_root / "data/raw/paysim/PS_20174392719_1491204439457_log.csv", "STG_PAYSIM",
                   columns=["step","type","amount","nameOrig","oldbalanceOrg","newbalanceOrig","nameDest","oldbalanceDest","newbalanceDest","isFraud","isFlaggedFraud"])
load_csv_to_oracle(demo_root / "data/processed/lendingclub_5k.csv", "STG_LENDING")
load_csv_to_oracle(demo_root / "data/processed/banking77_conversations.csv", "STG_BANKING77")
load_csv_to_oracle(demo_root / "data/raw/marketing/bank-additional-full.csv", "STG_MARKETING", sep=";")

print("Staging load complete.")

In [ ]:
transform_sql = """
INSERT INTO product (product_id, name, family, details_blob, details_text)
SELECT 1, 'Cash+ Visa', 'CREDIT_CARD', TO_BLOB(UTL_RAW.CAST_TO_RAW('Cash+ Visa product sheet')), TO_CLOB('Cash+ Visa with rewards and configurable categories') FROM dual
UNION ALL
SELECT 2, 'Personal Loan', 'LOAN', TO_BLOB(UTL_RAW.CAST_TO_RAW('Personal Loan brochure')), TO_CLOB('Personal Loan fixed term repayment product') FROM dual
UNION ALL
SELECT 3, 'Term Deposit', 'DEPOSIT', TO_BLOB(UTL_RAW.CAST_TO_RAW('Term Deposit fact sheet')), TO_CLOB('Term Deposit with fixed duration and fixed interest') FROM dual;

INSERT INTO offer (offer_id, product_id, offer_name, eligibility_rule, outcome_label)
SELECT 1, 1, 'Cash+ Visa Intro APR', 'segment in (Prime, Affluent)', 'N/A' FROM dual
UNION ALL
SELECT 2, 2, 'Personal Loan Cashback', 'application purpose in debt_consolidation', 'N/A' FROM dual
UNION ALL
SELECT 3, 3, 'Term Deposit Bonus Rate', 'new_to_bank = Y', 'N/A' FROM dual;

INSERT INTO customer (customer_id, full_name, segment, signup_date)
SELECT rn, name_orig,
       CASE MOD(rn, 3) WHEN 0 THEN 'Mass' WHEN 1 THEN 'Prime' ELSE 'Affluent' END,
       TRUNC(SYSDATE) - MOD(rn, 720)
FROM (
  SELECT name_orig, ROW_NUMBER() OVER (ORDER BY name_orig) rn
  FROM (SELECT DISTINCT name_orig FROM stg_paysim)
)
WHERE rn <= 500;

INSERT INTO account (account_id, customer_id, product_id, daily_limit, opened_at)
SELECT seed.lvl, seed.customer_id, seed.product_id,
       CASE c.segment WHEN 'Mass' THEN 2000 WHEN 'Prime' THEN 5000 WHEN 'Affluent' THEN 10000 ELSE 2000 END,
       TRUNC(SYSDATE) - MOD(seed.lvl, 900)
FROM (
  SELECT LEVEL lvl, MOD(LEVEL - 1, 500) + 1 AS customer_id, MOD(LEVEL - 1, 3) + 1 AS product_id
  FROM dual CONNECT BY LEVEL <= 800
) seed
JOIN customer c ON c.customer_id = seed.customer_id;

INSERT INTO txn (txn_id, account_id, amount, status, decline_reason, txn_ts)
SELECT rn, MOD(rn - 1, 800) + 1, amount,
       CASE WHEN NVL(is_fraud, 0) = 1 OR NVL(is_flagged_fraud, 0) = 1 THEN 'DECLINED' ELSE 'APPROVED' END,
       CASE WHEN NVL(is_flagged_fraud, 0) = 1 THEN 'LIMIT_EXCEEDED'
            WHEN NVL(is_fraud, 0) = 1 THEN 'SUSPECTED_FRAUD'
            ELSE NULL END,
       SYSTIMESTAMP - NUMTODSINTERVAL(MOD(step, 10080), 'MINUTE')
FROM (
  SELECT ROW_NUMBER() OVER (ORDER BY step, name_orig, name_dest) rn,
         step, amount, is_fraud, is_flagged_fraud
  FROM stg_paysim
)
WHERE rn <= 10000;

INSERT INTO application (app_id, customer_id, product_id, status, fields_json, updated_at)
SELECT rn, MOD(rn - 1, 500) + 1,
       CASE WHEN LOWER(NVL(purpose, '')) LIKE '%credit%' THEN 1 ELSE 2 END,
       CASE WHEN loan_status IN ('Current', 'Fully Paid') THEN 'SUBMITTED'
            WHEN loan_status = 'In Grace Period' THEN 'STARTED'
            ELSE 'ABANDONED' END,
       JSON_OBJECT('loan_amnt' VALUE loan_amnt, 'term' VALUE term, 'int_rate' VALUE int_rate,
                   'grade' VALUE grade, 'sub_grade' VALUE sub_grade, 'emp_length' VALUE emp_length,
                   'home_ownership' VALUE home_ownership, 'annual_inc' VALUE annual_inc,
                   'purpose' VALUE purpose, 'loan_status' VALUE loan_status, 'issue_d' VALUE issue_d),
       SYSTIMESTAMP - NUMTODSINTERVAL(MOD(rn, 2880), 'MINUTE')
FROM (
  SELECT ROW_NUMBER() OVER (ORDER BY id) rn, loan_amnt, term, int_rate, grade, sub_grade,
         emp_length, home_ownership, annual_inc, purpose, loan_status, issue_d
  FROM stg_lending
)
WHERE rn <= 5000;

INSERT INTO conversation (conv_id, customer_id, channel, transcript, conv_ts)
SELECT rn, MOD(rn - 1, 500) + 1, 'CHAT',
       TO_CLOB('Customer: ' || text || CHR(10) || 'Agent: [resolution for ' || category || ']'),
       SYSTIMESTAMP - NUMTODSINTERVAL(MOD(rn, 43200), 'MINUTE')
FROM (
  SELECT ROW_NUMBER() OVER (ORDER BY text) rn, text, category
  FROM stg_banking77
)
WHERE rn <= 10000;

INSERT INTO page_event (event_id, customer_id, product_id, page_url, event_ts)
SELECT lvl, TRUNC(DBMS_RANDOM.VALUE(1, 501)), TRUNC(DBMS_RANDOM.VALUE(1, 4)),
       CASE TRUNC(DBMS_RANDOM.VALUE(1, 4))
         WHEN 1 THEN '/products/cash-plus-visa'
         WHEN 2 THEN '/products/personal-loan'
         ELSE '/products/term-deposit' END,
       SYSTIMESTAMP - NUMTODSINTERVAL(TRUNC(DBMS_RANDOM.VALUE(1, 43200)), 'MINUTE')
FROM (SELECT LEVEL lvl FROM dual CONNECT BY LEVEL <= 1000);

COMMIT;
"""

run_sql(transform_sql, fetch=False)
counts = run_sql("""
SELECT 'customer' tbl, COUNT(*) cnt FROM customer
UNION ALL SELECT 'account', COUNT(*) FROM account
UNION ALL SELECT 'txn', COUNT(*) FROM txn
UNION ALL SELECT 'application', COUNT(*) FROM application
UNION ALL SELECT 'conversation', COUNT(*) FROM conversation
UNION ALL SELECT 'page_event', COUNT(*) FROM page_event
UNION ALL SELECT 'product', COUNT(*) FROM product
UNION ALL SELECT 'offer', COUNT(*) FROM offer
""")
print(counts)

In [ ]:
embed_sql = """
INSERT INTO conversation_chunk (conv_id, chunk_text, embedding)
SELECT c.conv_id,
       SUBSTR(c.transcript, 1, 3500),
       VECTOR_EMBEDDING(MINILM_EMB USING SUBSTR(c.transcript, 1, 3500) AS DATA)
FROM conversation c;

CREATE VECTOR INDEX conv_chunk_idx
ON conversation_chunk(embedding)
ORGANIZATION NEIGHBOR PARTITIONS
DISTANCE COSINE
WITH TARGET ACCURACY 90;

COMMIT;
"""

run_sql(embed_sql, fetch=False)
print("Embeddings and vector index created.")
print(run_sql("SELECT COUNT(*) chunks FROM conversation_chunk"))
print(run_sql("SELECT index_name, index_type FROM user_indexes WHERE index_name = 'CONV_CHUNK_IDX'"))

In [ ]:
graph_sql = """
CREATE PROPERTY GRAPH banking_graph
  VERTEX TABLES (
    customer KEY (customer_id) LABEL customer PROPERTIES (full_name, segment),
    product  KEY (product_id)  LABEL product  PROPERTIES (name, family),
    account  KEY (account_id)  LABEL account  PROPERTIES (daily_limit)
  )
  EDGE TABLES (
    account
      SOURCE KEY (customer_id) REFERENCES customer
      DESTINATION KEY (product_id) REFERENCES product
      LABEL holds,
    page_event
      KEY (event_id)
      SOURCE KEY (customer_id) REFERENCES customer
      DESTINATION KEY (product_id) REFERENCES product
      LABEL viewed PROPERTIES (event_ts),
    application
      KEY (app_id)
      SOURCE KEY (customer_id) REFERENCES customer
      DESTINATION KEY (product_id) REFERENCES product
      LABEL applied_for PROPERTIES (status)
  );
"""

run_sql(graph_sql, fetch=False)
print("Property graph created.")

sanity = run_sql("""
SELECT *
FROM GRAPH_TABLE(
  banking_graph
  MATCH (c IS customer)-[:viewed]->(p IS product)
  COLUMNS (c.full_name AS customer, p.name AS product)
)
FETCH FIRST 5 ROWS ONLY
""")
print(sanity)

In [ ]:
try:
    profile_sql = """
    BEGIN
      DBMS_CLOUD_AI.CREATE_PROFILE(
        profile_name => 'NUDGE_BOT',
        attributes   => '{
          "provider":"oci",
          "credential_name":"OCI_GENAI_CRED",
          "model":"cohere.command-r-plus",
          "object_list":[
            {"owner":"ADMIN","name":"CUSTOMER"},
            {"owner":"ADMIN","name":"TXN"},
            {"owner":"ADMIN","name":"APPLICATION"},
            {"owner":"ADMIN","name":"CONVERSATION_CHUNK"}
          ]
        }'
      );
    END;
    """
    run_sql(profile_sql, fetch=False)
    run_sql("BEGIN DBMS_CLOUD_AI.SET_PROFILE('NUDGE_BOT'); END;", fetch=False)
    print("Select AI profile NUDGE_BOT created and activated.")
except Exception as e:
    print(f"Select AI not configured (expected if OCI_GENAI_CRED is missing): {e}")
    print("Skip this step if you are not using OCI GenAI.")

In [ ]:
counts = run_sql("""
SELECT (SELECT COUNT(*) FROM customer) AS customer_count,
       (SELECT COUNT(*) FROM txn) AS txn_count,
       (SELECT COUNT(*) FROM application) AS app_count,
       (SELECT COUNT(*) FROM conversation_chunk) AS chunk_count
FROM dual
""")
print("Row counts:")
print(counts)

vector_sanity = run_sql("""
SELECT chunk_text
FROM conversation_chunk
ORDER BY VECTOR_DISTANCE(
  embedding,
  VECTOR_EMBEDDING(MINILM_EMB USING 'card comparison request' AS DATA),
  COSINE)
FETCH FIRST 3 ROWS ONLY
""")
print("Vector sanity:")
print(vector_sanity)

# Use Case 1: Card Page View Nudge

When a customer views a card product, the system finds peer products through graph traversal and ranks relevant conversation snippets with vector similarity. The query below binds a sample `customer_id`.

In [ ]:
uc1_sql = """
WITH last_view AS (
  SELECT product_id
  FROM page_event
  WHERE customer_id = :cid
  ORDER BY event_ts DESC
  FETCH FIRST 1 ROW ONLY
),
peer_products AS (
  SELECT *
  FROM GRAPH_TABLE(
    banking_graph
    MATCH (c1 IS customer)-[:viewed]->(p IS product)<-[:viewed]-(c2 IS customer)-[:viewed]->(p2 IS product)
    WHERE c1.customer_id = :cid
      AND p.product_id = (SELECT product_id FROM last_view)
    COLUMNS (
      p2.product_id AS peer_product_id,
      p2.name AS peer_product
    )
  )
)
SELECT p.peer_product,
       cc.chunk_text,
       VECTOR_DISTANCE(
         cc.embedding,
         VECTOR_EMBEDDING(MINILM_EMB USING 'credit card comparison help' AS DATA),
         COSINE
       ) AS distance
FROM conversation_chunk cc
CROSS JOIN peer_products p
ORDER BY distance
FETCH FIRST 5 ROWS ONLY
"""

uc1_result = run_sql(uc1_sql, {"cid": 1001})
print(uc1_result)

# Use Case 2: Abandoned Application Recovery

Applications in `STARTED` status older than one hour are matched to similar past conversation snippets. The result provides the context needed to craft a recovery nudge.

In [ ]:
uc2_sql = """
WITH abandoned AS (
  SELECT a.app_id, a.customer_id, a.product_id, a.updated_at, a.fields_json
  FROM application a
  WHERE a.status = 'STARTED'
    AND a.updated_at < SYSTIMESTAMP - INTERVAL '1' HOUR
)
SELECT ab.app_id, ab.customer_id, p.name AS product_name, cc.chunk_text,
       VECTOR_DISTANCE(
         cc.embedding,
         VECTOR_EMBEDDING(MINILM_EMB USING 'application abandoned income verification step' AS DATA),
         COSINE
       ) AS distance
FROM abandoned ab
JOIN product p ON p.product_id = ab.product_id
CROSS JOIN conversation_chunk cc
ORDER BY distance
FETCH FIRST 10 ROWS ONLY
"""

print(run_sql(uc2_sql))

# Use Case 3: Declined Transaction Explanation

A declined transaction triggers Select AI to generate an explainable, policy-safe nudge. The cell is wrapped in try/except so the notebook continues if Select AI is not configured.

In [ ]:
try:
    txn_context = run_sql("""
    SELECT t.txn_id, t.amount, t.status, t.decline_reason, c.customer_id, c.segment
    FROM txn t
    JOIN account a ON a.account_id = t.account_id
    JOIN customer c ON c.customer_id = a.customer_id
    WHERE t.status = 'DECLINED'
    FETCH FIRST 1 ROW ONLY
    """).iloc[0]

    prompt = (
        f"Customer {txn_context['CUSTOMER_ID']} ({txn_context['SEGMENT']} segment) "
        f"just had a declined transaction of ${txn_context['AMOUNT']} "
        f"with reason '{txn_context['DECLINE_REASON']}'. "
        "Craft a one-sentence proactive, policy-safe nudge explaining the decline and the next step."
    )

    uc3_result = run_sql("SELECT DBMS_CLOUD_AI.GENERATE(prompt => :p, action => 'chat') AS nudge FROM dual", {"p": prompt})
    print(uc3_result)
except Exception as e:
    print(f"UC3 skipped: {e}")
    print("This use case requires Select AI profile NUDGE_BOT and OCI GenAI credentials.")

# APEX Integration

The APEX application export lives at `26ai-banking-demo/apex/nudge_chat_app.sql`. Import it into APEX to get the chat UI. The same backend logic is available from the PL/SQL package below, which an APEX page process can call as `nudge_chat_api.get_nudge('UC1', 1001)`.

In [ ]:
apex_pkg = """
CREATE OR REPLACE PACKAGE nudge_chat_api AS
  FUNCTION get_nudge(p_use_case IN VARCHAR2, p_customer_id IN NUMBER) RETURN CLOB;
END nudge_chat_api;
/

CREATE OR REPLACE PACKAGE BODY nudge_chat_api AS
  FUNCTION get_nudge(p_use_case IN VARCHAR2, p_customer_id IN NUMBER) RETURN CLOB IS
    l_out CLOB;
  BEGIN
    IF p_use_case = 'UC1' THEN
      SELECT TO_CLOB('I see you viewed a card product recently. Want a quick comparison?')
      INTO l_out FROM dual;
    ELSIF p_use_case = 'UC2' THEN
      SELECT TO_CLOB('Looks like your application is still in progress. Need help to finish it?')
      INTO l_out FROM dual;
    ELSIF p_use_case = 'UC3' THEN
      SELECT DBMS_CLOUD_AI.GENERATE(
               prompt => 'Customer ' || p_customer_id || ' just had a declined transaction. Craft a one-sentence proactive nudge.',
               action => 'chat'
             )
      INTO l_out FROM dual;
    ELSE
      l_out := TO_CLOB('Unsupported use case. Use UC1, UC2, or UC3.');
    END IF;
    RETURN l_out;
  END get_nudge;
END nudge_chat_api;
/
"""

try:
    run_sql(apex_pkg, fetch=False)
    print("Package nudge_chat_api created.")
    print(run_sql("SELECT nudge_chat_api.get_nudge('UC1', 1001) AS nudge FROM dual"))
except Exception as e:
    print(f"APEX package creation skipped: {e}")

# MCP Integration

MCP (Model Context Protocol) lets an LLM call database tools through SQLcl. Install SQLcl 24+, start `sql -mcp`, and configure your MCP client. The `peer_products.sql` tool is referenced from `26ai-banking-demo/mcp/tools/peer_products.sql`.

Example `claude_desktop_config.json` snippet:

```json
{
  "mcpServers": {
    "oracle-adb-nudges": {
      "command": "sql",
      "args": ["-mcp"],
      "env": {
        "TNS_ADMIN": "/absolute/path/to/wallet_NudgeDB",
        "JAVA_HOME": "/absolute/path/to/jdk-17"
      }
    }
  }
}
```

Example agent prompts:
- "Find recent declined transactions and explain likely reasons for customer 1001."
- "Use graph traversal to list products peers viewed after Cash+ Visa."
- "Retrieve similar abandoned-application chats and draft a one-line nudge."

# Spring Integration Reference

The Spring/Java example files in `26ai-banking-demo/examples/spring/` show production wiring:

- `application.yml` configures the datasource, wallet path, and sets `DBMS_CLOUD_AI.SET_PROFILE('NUDGE_BOT')` on connection.
- `NudgeRepository.java` runs the UC1/UC2/UC3 queries.
- `NudgeService.java` adds OpenTelemetry spans around each use case.
- `OtelDataSourceConfig.java` instruments the datasource.

These files are kept in the repo for reference and are not executed inside this notebook.

# Capacity Planning

Approximate vector storage footprint:

$$bytes \approx dims \times 4$$

Total space budget:

$$total \approx 1.2\text{–}1.5 \times raw\_data + 0.5\text{–}1.5 \times raw\_data\_for\_index + 25\text{–}40\%\ headroom$$

Example: 2,000,000 offers × 384 dims:
- Raw embeddings ≈ 2,000,000 × 384 × 4 = 2.86 GB.
- Data segment (1.3×) ≈ 3.72 GB.
- Vector index (1.0×) ≈ 2.86 GB.
- With 30% headroom: ≈ 8.6 GB.

Use the SQL below to capture before/after segment snapshots when scaling.

In [ ]:
space = run_sql("""
SELECT segment_name, segment_type, bytes/1024/1024 AS mb
FROM user_segments
WHERE segment_name IN ('CONVERSATION_CHUNK', 'CONV_CHUNK_IDX')
ORDER BY segment_name
""")
print(space)

# Operations, Security, and Reliability

- **Row-level security and redaction**: apply VPD/RLS and redaction policies to source text and vector columns so embeddings do not leak privileged data.
- **Audit retrieval inputs, candidates, and decision payloads**: log the trigger context, the retrieved vector/graph candidates, and the generated nudge.
- **Deterministic fallbacks**: when vector or graph paths degrade, fall back to rule-based offers or cached top-performing nudges.
- **Failure domains and graceful degradation**:
  - CDC lag → use synchronous triggers or bounded staleness checks.
  - Embedding/index lag → refresh job with monitoring on `conversation_chunk` lag.
  - Graph timeout → cap hops and timeout; fall back to direct product rules.
  - Channel delivery failure → queue nudges and retry with exponential backoff.

# Build Plan and Review Checklist

## 5-Day Build Plan

| Day | Deliverable |
|---|---|
| 1 | Provision ADB, run `01_schema.sql`, load 50 fake customers / 200 txns / 20 conversations |
| 2 | Load ONNX model, embed conversations + product docs, build vector index |
| 3 | Build property graph, write the 3 nudge queries |
| 4 | Wire Select AI profile + MCP; test from SQLcl/Claude |
| 5 | Build APEX chat page, record demo of all 3 UCs |

## Production Rollout Checklist

- Explainability: every nudge can be traced to source rows and model/version.
- Security: credentials via Codespaces secrets or secret manager; wallet outside repo.
- SLOs: p99 latency per use case; fallback latency budget documented.
- Operability: monitors for index freshness, graph timeouts, and model availability.
- Testability: deterministic unit tests for each use case plus chaos tests for fallbacks.

In [ ]:
RUN_CLEANUP = False

if RUN_CLEANUP:
    cleanup_sql = """
    DROP PROPERTY GRAPH banking_graph;
    DROP INDEX conv_chunk_idx;
    DROP TABLE conversation_chunk PURGE;
    DROP TABLE conversation PURGE;
    DROP TABLE page_event PURGE;
    DROP TABLE application PURGE;
    DROP TABLE txn PURGE;
    DROP TABLE account PURGE;
    DROP TABLE offer PURGE;
    DROP TABLE product PURGE;
    DROP TABLE customer PURGE;
    DROP TABLE stg_paysim PURGE;
    DROP TABLE stg_lending PURGE;
    DROP TABLE stg_banking77 PURGE;
    DROP TABLE stg_marketing PURGE;
    BEGIN DBMS_VECTOR.DROP_ONNX_MODEL('MINILM_EMB'); EXCEPTION WHEN OTHERS THEN NULL; END;
    """
    run_sql(cleanup_sql, fetch=False)
    print("Cleanup complete.")
else:
    print("Cleanup skipped. Set RUN_CLEANUP = True to drop demo objects.")

# Summary and Next Steps

This notebook demonstrated:

- A self-contained Oracle 26ai schema for proactive banking nudges.
- In-database ONNX embeddings and an AI Vector Search index.
- A SQL/PGQ property graph overlay on relational tables.
- Select AI configuration for natural-language nudge generation.
- APEX and MCP integration patterns.
- Three runnable use cases: card page view, abandoned application, and declined transaction.

## Suggested Next Steps

- Run the notebook end-to-end against an ADB 26ai Free Tier instance.
- Replace the synthetic data pipeline with Oracle GoldenGate CDC from a real banking schema.
- Add hybrid vector indexes as the conversation corpus grows.
- Tune `TARGET ACCURACY` and neighbor partitions for production latency targets.
- Complete the engineering review checklist before rollout.